# Overfitting e hiperparametros


Usa esta plantilla cuando pidan mejorar un modelo, comparar train/test, usar cross-validation o ajustar hiperparámetros.

La idea central: si el modelo va muy bien en train y mal fuera, está memorizando.


In [ ]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

TARGET = "target"  # CAMBIAR

df = pd.read_csv("data.csv")
X = df.drop(columns=[TARGET])
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y if y.nunique() > 1 else None,
)

numeric_cols = X_train.select_dtypes(include="number").columns
categorical_cols = X_train.select_dtypes(exclude="number").columns

preprocessor = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), numeric_cols),
        ("cat", Pipeline([("imputer", SimpleImputer(strategy="most_frequent")), ("encoder", OneHotEncoder(handle_unknown="ignore"))]), categorical_cols),
    ]
)


In [ ]:
base_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42)),
    ]
)

base_model.fit(X_train, y_train)

print("TRAIN")
print(classification_report(y_train, base_model.predict(X_train), zero_division=0))

print("TEST")
print(classification_report(y_test, base_model.predict(X_test), zero_division=0))


In [ ]:
# Ajuste sencillo de hiperparámetros.
# Mantén la grid pequeña: en examen interesa demostrar criterio, no probar 200 combinaciones.
model_for_search = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", RandomForestClassifier(random_state=42)),
    ]
)

param_grid = {
    "classifier__n_estimators": [100, 200],
    "classifier__max_depth": [None, 8, 16],
    "classifier__min_samples_leaf": [1, 3, 5],
}

search = GridSearchCV(
    model_for_search,
    param_grid=param_grid,
    cv=5,
    scoring="f1_macro",
    n_jobs=-1,
)

search.fit(X_train, y_train)

print("Mejores parámetros:", search.best_params_)
print("Mejor score CV:", search.best_score_)

best_model = search.best_estimator_
print(classification_report(y_test, best_model.predict(X_test), zero_division=0))


## Cómo diagnosticar

- Train alto y test bajo: overfitting → sube `min_samples_leaf`, baja `max_depth`, más datos.
- Train bajo y test bajo: underfitting → modelo demasiado simple, añade features o sube complejidad.
- CV alto y test bajo: posible leakage o el test tiene distribución distinta.
- Una clase va mal: cambia `scoring` a `f1_macro`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import learning_curve

# Curva de aprendizaje — muestra overfitting/underfitting según el tamaño del dataset
# Usar el modelo base (sin ajustar) para ver el diagnóstico limpio
train_sizes, train_scores, val_scores = learning_curve(
    base_model,
    X_train, y_train,
    cv=5,
    scoring='f1_macro',
    train_sizes=np.linspace(0.1, 1.0, 8),
    n_jobs=-1,
)

train_mean = train_scores.mean(axis=1)
val_mean   = val_scores.mean(axis=1)
train_std  = train_scores.std(axis=1)
val_std    = val_scores.std(axis=1)

plt.figure(figsize=(7, 4))
plt.plot(train_sizes, train_mean, 'o-', label='Train')
plt.plot(train_sizes, val_mean,   's-', label='Validación CV')
plt.fill_between(train_sizes, train_mean - train_std, train_mean + train_std, alpha=0.15)
plt.fill_between(train_sizes, val_mean   - val_std,   val_mean   + val_std,   alpha=0.15)
plt.xlabel('Muestras de entrenamiento')
plt.ylabel('F1 macro')
plt.title('Curva de aprendizaje')
plt.legend()
plt.tight_layout()
plt.show()

# Interpretación:
# Train >> Val con gap grande → OVERFITTING (modelo demasiado complejo)
# Train ≈ Val ambos bajos    → UNDERFITTING (modelo demasiado simple)
# Train ≈ Val ambos altos    → BIEN generalizado